# Big Data project -> Steam Games Revies

### Proposal

I'd like to propose a possible project for the Big Data exam, I'd like to mix 2 kaggle datasets:
- https://www.kaggle.com/datasets/kieranpoc/steam-reviews
- https://www.kaggle.com/datasets/crainbramp/steam-dataset-2025-multi-modal-gaming-analytics

The proposal will analyse all reviews (data retrieved from the first dataset) and join them with games and genres (which are taken, instead, from the second dataset) in order to calculate the average hours played for positive vs negative reviews, broken down by genre. If the proposal is not hard enough I'm open to possible complications.


### Datasets

Review dataset columns structure:

recommendationid,appid,game,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,hidden_in_steam_china,steam_china_location

It will be reduced to:
recommendationid[0], appid[1], game[2], author_steamid[3], review[11], weighted_vote_score[17]

App dataset column structure:

appid,name,type,is_free,release_date,required_age,short_description,supported_languages,header_image,background,metacritic_score,recommendations_total,mat_supports_windows,mat_supports_mac,mat_supports_linux,mat_initial_price,mat_final_price,mat_discount_percent,mat_currency,mat_achievement_count,mat_pc_os_min,mat_pc_processor_min,mat_pc_memory_min,mat_pc_graphics_min,mat_pc_os_rec,mat_pc_processor_rec,mat_pc_memory_rec,mat_pc_graphics_rec,created_at,updated_at

reduced to:

appid[0], name[1], type[2]

Genre dataset column structure:

id, genre

### Dataset loading

In [1]:
reviewFilePath = 'datasets/small_reviews.csv'
applicationFilePath = 'datasets/sample_apps.csv'
genreFilePath = 'datasets/genres.csv'
genreByAppFilePath = 'datasets/application_genres_standardized.csv'

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .master("local[4]") \
    .appName("Local Spark") \
    .config('spark.ui.port', '4040') \
    .getOrCreate()
sc = spark.sparkContext

In [4]:
from typing import Optional, Tuple
from pyspark.sql import Row

class SteamParser:

    @staticmethod
    def parse_application_line(row) -> Optional[Tuple[int, str, int, int]]:
        try:
            parts = list(row)
            
            appid = int(parts[0]) if parts[0] else 0
            name = parts[1] if parts[1] else ""
            metacritic = int(parts[10]) if parts[10] else 0
            recommendations = int(parts[11]) if parts[11] else 0
            
            return (appid, name, metacritic, recommendations)
        except Exception as e:
            return None

    @staticmethod
    def parse_review_line(row) -> Optional[Tuple[int, bool, float, float, float]]:
        try:
            parts = list(row)
            
            appid = int(parts[1].strip())
            playtime_forever = float(parts[6].strip()) if parts[6].strip() else 0.0
            playtime_at_review = float(parts[8].strip()) if parts[8].strip() else 0.0
            voted_up = parts[14].strip().lower() == 'true'
            weight_vote_score = float(parts[17].strip())
            # Filter invalid data
            if playtime_at_review < 0:
                return None
            return (appid, voted_up, playtime_forever, playtime_at_review, weight_vote_score)
        except:
            return None

    @staticmethod
    def parse_genre_line(row) -> Optional[Tuple[int, str]]:
        try:
            parts = list(row)
            
            genre_id = int(parts[0].strip())
            genre_name = parts[1].strip()
            
            if not genre_name:
                return None
            return (genre_id, genre_name)
        except:
            return None

    @staticmethod
    def parse_app_genre_line(row) -> Optional[Tuple[int, int]]:
        try:
            parts = list(row)

            appid = int(parts[0].strip())
            genre_id = int(parts[1].strip())
            return (appid, genre_id)
        except:
            return None

In [6]:
# Games parsing
df = spark.read.csv(
    applicationFilePath,
    header=True,  # if your CSV has headers
    quote='"',
    escape='"',
    multiLine=True  # handles newlines within quoted fields
)

rddGames = df.rdd \
    .map(lambda row: SteamParser.parse_application_line(row)) \
    .filter(lambda x: x is not None)

# Reviews parsing
df = spark.read.csv(
    reviewFilePath,
    header=True,  # if your CSV has headers
    quote='"',
    escape='"',
    multiLine=True  # handles newlines within quoted fields
)

rddReviews = df.rdd \
    .map(lambda row: SteamParser.parse_review_line(row)) \
    .filter(lambda x: x is not None)

# Genres parsing
df = spark.read.csv(
    genreFilePath,
    header=True,  # if your CSV has headers
    quote='"',
    escape='"',
    multiLine=True  # handles newlines within quoted fields
)

rddGenres = df.rdd \
    .map(lambda row: SteamParser.parse_genre_line(row)) \
    .filter(lambda x: x is not None)


# Genres by App parsing
df = spark.read.csv(
    genreByAppFilePath,
    header=True,  # if your CSV has headers
    quote='"',
    escape='"',
    multiLine=True  # handles newlines within quoted fields
)

rddGenresByApp = df.rdd \
    .map(lambda row: SteamParser.parse_app_genre_line(row)) \
    .filter(lambda x: x is not None)

### Dataset exploration

In [7]:
print(rddReviews.first() if rddReviews else "")
print(rddGames.first() if rddGames else "")
print(rddGenres.first() if rddGenres else "")
print(rddGenresByApp.first() if rddGenresByApp else "")

(10, False, 4653.0, 4650.0, 0.0)
(3460, 'Talismania Deluxe', 0, 0)
(1, '360 Video')
(10, 3)


### Execution

In [8]:
num_partitions = 8

In [9]:
rddGames = rddGames.repartition(num_partitions).cache()
rddReviews = rddReviews.repartition(num_partitions).cache()
rddGenres = rddGenres.cache()  # Small dataset, keep cached
rddGenresByApp = rddGenresByApp.repartition(num_partitions).cache()

In [10]:
# Games: (appid, (name, metacritic, recommendations))
games_kv = rddGames.map(lambda x: (x[0], (x[1], x[2], x[3])))

# Reviews: (appid, (voted_up, playtime_at_review, weight_vote_score))
reviews_kv = rddReviews.map(lambda x: (x[0], (x[1], x[3], x[4])))

# GenresByApp: (appid, genre_id)
genres_by_app_kv = rddGenresByApp.map(lambda x: (x[0], x[1]))

# Genres: (genre_id, genre_name)
genres_kv = rddGenres.map(lambda x: (x[0], x[1]))

In [11]:
reviews_with_genre_id = reviews_kv.join(genres_by_app_kv)

In [12]:
reviews_by_genre_id = reviews_with_genre_id.map(
    lambda x: (x[1][1], (x[1][0][0], x[1][0][1], x[1][0][2]))
)

In [13]:
reviews_with_genre_name = reviews_by_genre_id.join(genres_kv)

In [14]:
genre_sentiment_pairs = reviews_with_genre_name.map(
    lambda x: ((x[1][1], x[1][0][0]), (x[1][0][1], x[1][0][2]))
)

In [15]:
# Aggregate: sum of weighted playtime and sum of weights
def aggregate_weighted_playtime(acc, value):
    """Accumulator function for weighted average calculation"""
    playtime, weight = value
    weighted_playtime = playtime * weight
    return (acc[0] + weighted_playtime, acc[1] + weight, acc[2] + 1)

def combine_aggregates(acc1, acc2):
    """Combiner function for weighted average calculation"""
    return (acc1[0] + acc2[0], acc1[1] + acc2[1], acc1[2] + acc2[2])

In [16]:
aggregated = genre_sentiment_pairs.aggregateByKey(
    (0.0, 0.0, 0),  # (sum_weighted_playtime, sum_weights, count)
    aggregate_weighted_playtime,
    combine_aggregates
)

In [17]:
results = aggregated.map(
    lambda x: (
        x[0][0],  # genre_name
        "Positive" if x[0][1] else "Negative",  # sentiment
        x[1][0] / x[1][1] if x[1][1] > 0 else 0.0,  # weighted_avg_hours
        x[1][2]  # review_count
    )
)

In [18]:
sorted_results = results.sortBy(lambda x: (x[0], x[1]))

# Collect results
final_results = sorted_results.collect()

In [19]:
print("\n" + "="*80)
print("AVERAGE HOURS PLAYED: POSITIVE VS NEGATIVE REVIEWS BY GENRE")
print("="*80)
print(f"{'Genre':<30} {'Sentiment':<12} {'Avg Hours':<15} {'Review Count':<15}")
print("-"*80)

for genre, sentiment, avg_hours, count in final_results:
    print(f"{genre:<30} {sentiment:<12} {avg_hours:<15.2f} {count:<15,}")

print("="*80)


AVERAGE HOURS PLAYED: POSITIVE VS NEGATIVE REVIEWS BY GENRE
Genre                          Sentiment    Avg Hours       Review Count   
--------------------------------------------------------------------------------
Action                         Negative     8779.66         18,615         
Adventure                      Negative     5797.99         11,296         
Animation & Modeling           Negative     7408.72         229            
Audio Production               Negative     6291.43         19             
Casual                         Negative     5651.89         4,172          
Design & Illustration          Negative     5120.34         227            
Early Access                   Negative     9500.02         985            
Education                      Negative     3398.83         16             
Episodic                       Negative     0.00            3              
Free to Play                   Negative     17553.84        6,688          
Game Development      

In [20]:
genre_comparison = {}
for genre, sentiment, avg_hours, count in final_results:
    if genre not in genre_comparison:
        genre_comparison[genre] = {}
    genre_comparison[genre][sentiment] = {"avg_hours": avg_hours, "count": count}

print("\n" + "="*80)
print("GENRE COMPARISON: POSITIVE VS NEGATIVE REVIEW HOURS")
print("="*80)
print(f"{'Genre':<30} {'Pos Hrs':<12} {'Neg Hrs':<12} {'Difference':<15} {'Total Reviews':<15}")
print("-"*80)

comparison_results = []
for genre, data in genre_comparison.items():
    pos_hours = data.get("Positive", {}).get("avg_hours", 0)
    neg_hours = data.get("Negative", {}).get("avg_hours", 0)
    pos_count = data.get("Positive", {}).get("count", 0)
    neg_count = data.get("Negative", {}).get("count", 0)
    difference = pos_hours - neg_hours
    total_reviews = pos_count + neg_count
    
    comparison_results.append((genre, pos_hours, neg_hours, difference, total_reviews))

# Sort by difference (descending)
comparison_results.sort(key=lambda x: x[3], reverse=True)

for genre, pos_hrs, neg_hrs, diff, total in comparison_results:
    print(f"{genre:<30} {pos_hrs:<12.2f} {neg_hrs:<12.2f} {diff:<+15.2f} {total:<15,}")

print("="*80)


GENRE COMPARISON: POSITIVE VS NEGATIVE REVIEW HOURS
Genre                          Pos Hrs      Neg Hrs      Difference      Total Reviews  
--------------------------------------------------------------------------------
Episodic                       0.00         0.00         +0.00           3              
Movie                          0.00         0.00         +0.00           3              
Short                          0.00         0.00         +0.00           2              
Sexual Content                 0.00         128.67       -128.67         3              
Violent                        0.00         146.42       -146.42         21             
Nudity                         0.00         191.80       -191.80         2              
Gore                           0.00         200.43       -200.43         11             
Education                      0.00         3398.83      -3398.83        16             
Photo Editing                  0.00         4058.36      -4058.36